# Retrieval Augmented Generation
In this notebook, we'll showcase how to build an AI agent that implements retrieval augmented generation powered by a Vector Database. This will allow you to feed external data sources (particularly large documents that would exceed LLM context windows) into your agent and get responses grounded on your ingested data sources (the source of truth)

In [7]:
import os
import bs4
import requests
import langsmith
from dotenv import load_dotenv

from langchain.tools import Tool
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader

In [8]:
load_dotenv()

# LLM API configuration
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# LangSmith configuration
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "true")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "default")
LANGSMITH_WORKSPACE_ID = os.getenv("LANGSMITH_WORKSPACE_ID")

## Building our Retrieval Source

### Embedding Model & Vector Database
- Instantiate our **embedding model** to convert our text into a vector space; 
- and our **vector store** where we'll store our vector embeddings.

In [9]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001") # Embeddings model
vector_store = InMemoryVectorStore(embeddings) # In-memory vector store to hold document embeddings

### Document Loaders & Chunking
Load data from our actual sources. In this tutorial we'll only covered 3 types of documents:
- Standard Text Documents
- Web-based Documents
- PDF-based Documents

Once a document is loaded, we'll split it into chunks and load every chunk into our vector store.

#### Loading PDF Files
Here we'll pass the Kenyan constitution as our first document, available in PDF format.

In [14]:
# Load a PDF document
loader = PyPDFLoader("../sources/kenyan-constitution.pdf")
docs = loader.load()

print(f'''Number of docs: {len(docs)}. Each doc is a page from the PDF. 
      Now, let's see some sample content from the first 5 docs:''')

for i in range(5):
    print(f"\n--- Doc {i+1} ---")
    print(docs[i].page_content)

Number of docs: 193. Each doc is a page from the PDF. 
      Now, let's see some sample content from the first 5 docs:

--- Doc 1 ---
LAWS OF KENYA
THE CONSTITUTION OF KENYA, 2010
Published by the National Council for Law Reporting
with the Authority of the Attorney-General
www.kenyalaw.org

--- Doc 2 ---
Constitution of Kenya, 2010
THE CONSTITUTION OF KENYA, 2010
ARRANGEMENT OF ARTICLES
PREAMBLE
CHAPTER ONE—SOVEREIGNTY OF THE PEOPLE AND  
SUPREMACY OF THIS CONSTITUTION
1—Sovereignty of the people.
2—Supremacy of this Constitution.
3—Defence of this Constitution.
CHAPTER TWO—THE REPUBLIC
4—Declaration of the Republic.
5—Territory of Kenya.
6—Devolution and access to services.
7—National, official and other languages.
8—State and religion.
9—National symbols and national days.
10—National values and principles of governance.
11—Culture.
CHAPTER THREE—CITIZENSHIP
12—Entitlements of citizens.
13—Retention and acquisition of citizenship.
14—Citizenship by birth.
15—Citizenship by registrat

In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 458 sub-documents.


In [16]:
document_ids = vector_store.add_documents(documents=all_splits)
print(document_ids)

['13be87bf-1ce9-4b6f-a042-53084bb3f629', 'c6045edc-e731-4c7e-bd0e-6bf0b3dc65b1', '01902847-d480-4cac-b763-a262d7055d11', '323dd4fd-c948-457a-a114-0a73e31e0302', '7bb6530f-ea0b-4ad2-959e-b9f36f3a5a7f', '3f339f76-e47e-46b2-ba70-510f477fdd5a', 'e0795de5-a7c9-47a4-81c3-d5c00e823b8f', '5ff9697f-d73b-46cf-8e73-3afeade41a2c', '3ecbb2d9-418d-4195-8b2c-bc7c4f1d7d62', '834e47bb-af0b-4966-be74-e42bf6d3fa47', '3f139b40-9ff4-48bd-929a-f7de79dcfa5a', '0d28a3f9-d01b-473c-9e3d-a3c5d3695622', '4277a2bb-8da8-48c9-9c35-19aabad6ddbe', 'ca10e85d-6bdc-41c8-bc1c-b9a9e4f23e30', 'aaa2d749-1319-49af-8049-43ee78c013ce', 'f6d218d3-0ea8-4dbf-a2d3-eb634bb9374b', '54fa74eb-5a9c-4923-ac69-514aaf5dc1dd', '4f70abc1-dd65-4646-bf15-644e940f724b', '35d90972-6320-4fa2-9b65-43d2e7827da6', '8720f40d-4845-47d1-b2fe-61cabe88c0a8', '7da91d12-08c9-4da8-b774-2d5262b62d59', '6ab4f74f-8dd6-4d3d-acc8-04bc65a5b942', '1f319b6f-b6c0-41ad-acff-07b2050ce86c', '3c594b6f-7d29-4cc3-8e83-ba0820f56149', '7588bdfb-a14c-41e8-b68c-035e18bb1753',

## Generate AI Responses grounded on our RAG Sources

In [17]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=5,
)

In [20]:
def retrieve_and_generate(query: str, k: int = 5):
    # Retrieve relevant documents
    results = vector_store.similarity_search(query, k=k)
    
    # Create context from retrieved documents
    context = "\n\n".join([doc.page_content for doc in results])
    
    # Create messages with context
    messages = [
        SystemMessage(content=f'''You are an expert assistant. Use the provided context to answer the user's question accurately and comprehensively. 
        If the answer cannot be found in the context, say so clearly.
        
        Context:
        {context}'''),
        HumanMessage(content=query)
    ]
    
    # Generate response
    response = model.invoke(messages)
    return response.content

# Example usage
user_query = input("Enter your question about the documents: ")
answer = retrieve_and_generate(user_query)
print(f'''{user_query} \n Answer based on the documents:" \n {answer}''')

how long can I be held by the police 
 Answer based on the documents:" 
 Based on the provided context, an arrested person has the right to be brought before a court as soon as reasonably possible, but not later than:

*   **Twenty-four hours** after being arrested.
*   If the twenty-four hours ends outside ordinary court hours, or on a day that is not an ordinary court day, then by the **end of the next court day**.
